Kelly nécessite des odds (cotes).
Pour l’instant ton SofaScore RapidAPI n’a pas d’endpoint odds, donc ici on prépare la logique et on simule avec une colonne odds_homewin si tu l’ajoutes plus tard (autre API / scraping légal / dataset).

load + kelly utils

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent / "src"))

import numpy as np
import pandas as pd

from config import get_settings
from data.storage import read_parquet
from betting.kelly import stake
from betting.strategy import should_bet

s = get_settings()
df = read_parquet(s.data_dir / "reports" / "train_frame.parquet")

df.head()

exemple “si on avait les cotes”

In [ ]:
# EXEMPLE: tu simules des odds pour tester le pipeline
# plus tard tu remplaceras par de vraies odds
rng = np.random.default_rng(42)
df = df.copy()
df["odds_homewin"] = 1.6 + rng.random(len(df)) * 1.4  # entre 1.6 et 3.0 (fake)

bankroll = 1000.0
curve = []

for _, r in df.iterrows():
    p = float(r["oof_proba"])          # proba modèle
    odds = float(r["odds_homewin"])    # cote
    y = int(r["y_homewin"])            # résultat réel

    if should_bet(p, odds, margin=0.01):
        amt = stake(bankroll, p, odds, kelly_mult=0.25, max_frac=0.05)
        if amt > 0:
            if y == 1:
                bankroll += amt * (odds - 1.0)
            else:
                bankroll -= amt

    curve.append(bankroll)

curve[-1], min(curve), max(curve)